In [ ]:
#| default_exp gsc.filters

# GSC filters
> Query, filter, and persist Google Search Console analytics data in SQLite.

In [ ]:
#| export
from seootter.models import GSCAnalytics
from sqlalchemy import func, not_, or_
from sqlmodel import Session

In [ ]:
# | export
def filter_site(query,  # SQLModel query
                site_url: str  # GSC property URL
                ):
    "Filter query by site URL."
    return query.where(GSCAnalytics.site_url == site_url)


def filter_dates(query,  # SQLModel query
                 start: str,  # Start date (YYYY-MM-DD)
                 end: str  # End date (YYYY-MM-DD)
                 ):
    "Filter query by date range."
    return query.where(GSCAnalytics.date.between(start, end))


def filter_dimension(query,  # SQLModel query
                     dimension: str,  # GSCAnalytics field name
                     value: str  # Value to match
                     ):
    "Filter query by a specific dimension value."
    return query.where(getattr(GSCAnalytics, dimension) == value)


def filter_exclude_pages(query,  # SQLModel query
                         exclude_pages: list[str]  # Page path substrings to exclude
                         ):
    "Exclude rows where page contains any of the given substrings."
    return query.where(not_(or_(*[GSCAnalytics.page.contains(p) for p in exclude_pages])))



In [ ]:
# | export
from sqlmodel import SQLModel


class AnalyticsSummary(SQLModel):
    query: str
    clicks: int
    impressions: int

In [ ]:
#| export
from urllib.parse import unquote
from sqlalchemy.dialects.sqlite import insert


def normalize_url(url: str) -> str:
    "Normalize URL by decoding percent-encoding and standardizing separators."
    url = unquote(url)
    url = url.replace("-", " ").replace("_", " ")
    return url


def parse_gsc_row(row: dict,  # Raw GSC API row
                  site_url: str,  # GSC property URL
                  date: str  # Date of the row (YYYY-MM-DD)
                  ) -> GSCAnalytics:
    "Parse a raw GSC API row into a GSCAnalytics instance."
    dims = dict(zip(("query", "page", "country", "device"), row["keys"]))
    return GSCAnalytics(site_url=site_url, date=date,
                        query=dims.get("query"), page=dims.get("page"),
                        country=dims.get("country"), device=dims.get("device"),
                        clicks=row["clicks"], impressions=row["impressions"],
                        ctr=row["ctr"], position=row["position"])


def store_gsc_data(session: Session,  # Active database session
                   site_url: str,  # GSC property URL
                   date: str,  # Date of the data (YYYY-MM-DD)
                   rows: list[dict]  # Raw GSC API rows
                   ) -> None:
    "Store GSC rows with upsert (update on conflict)."
    for row in rows:
        record = parse_gsc_row(row, site_url, date)
        stmt = insert(GSCAnalytics).values(**record.model_dump(exclude={"id"}))
        stmt = stmt.on_conflict_do_update(
            index_elements=["site_url", "date", "query", "page", "country", "device"],
            set_={"clicks": record.clicks, "impressions": record.impressions,
                  "ctr": record.ctr, "position": record.position}
        )
        session.exec(stmt)
    session.commit()


In [ ]:
#| export
from seootter.models import GSCPropertyTotals

def store_gsc_property_totals(session: Session,
                              site_url: str,
                              date: str,
                              rows: list[dict]
                              ) -> None:
    """Store site-level GSC totals with upsert."""
    if not rows:
        return
    row = rows[0]  # There should only be one row since we only group by date
    stmt = insert(GSCPropertyTotals).values(
        site_url=site_url, date=date,
        clicks=row["clicks"], impressions=row["impressions"],
        ctr=row["ctr"], position=row["position"]
    )
    stmt = stmt.on_conflict_do_update(
        index_elements=["site_url", "date"],
        set_={'clicks': row['clicks'], 'impressions': row['impressions'], 'ctr': row['ctr'], 'position': row['position']}
    )
    session.exec(stmt)
    session.commit()
